In [1]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.svm import SVR
from pandas import DataFrame, concat
from pickle import load
from sklearn.metrics import r2_score, root_mean_squared_error

In [3]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/potency.pickle', 'rb') as file:
    potency = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [4]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

In [5]:
X = descriptors_transformer.transform(molecules)
Y = potency['Potency']
XY = concat((X, Y), axis=1)

In [6]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [7]:
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [8]:
scaler_y = MinMaxScaler().fit(y_train.to_numpy().reshape(-1, 1))
y_train_scal = scaler_y.transform(y_train.to_numpy().reshape(-1, 1))
y_test_scal = scaler_y.transform(y_test.to_numpy().reshape(-1, 1))

In [ ]:
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
c = [1, 10, 50]
g = [0.1, 0.01, 0.001]
cf = [0, 0.5, 1]
epsilon = [0.01, 0.2, 0.3]
pg = [
    {
        'kernel': ['poly'],
        'gamma': g,
        'C': c, 
        'degree': [2, 3],
        'coef0' : cf,
        'epsilon': epsilon

    },
    {
        'kernel': ['sigmoid'],
        'gamma': g,
        'C' : c,
        'coef0' : cf,        
        'epsilon': epsilon
    },
    {
        'kernel':  ['rbf'],
        'gamma': g,
        'C' : c,
        'epsilon': epsilon

    }]
grid = GridSearchCV(SVR(verbose=3, shrinking=False), pg, cv=cv, verbose=3, scoring='r2', n_jobs=12)
grid.fit(x_train_scal, y_train_scal.ravel())

Fitting 25 folds for each of 270 candidates, totalling 6750 fits
.......................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [16]:
grid.best_estimator_

SVR(C=1, epsilon=0.2, gamma=0.1, shrinking=False, verbose=3)

In [17]:
y_pred = grid.predict(x_test_scal)
print(f'R^2 = {r2_score(y_test_scal, y_pred)}')
print(f'RMSE = {root_mean_squared_error(y_test_scal, y_pred)}')

R^2 = 0.13641528063874986
RMSE = 0.18697258054575192
